# Week 3, Day 3: Three tokenizers, one WrestleMania question

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/1WD6Y2N7ctQi1X9wa6rpkg8UfyA4iSVuz) linked from [week3/day3.ipynb](../../../week3/day3.ipynb), *Tokenizers*. **About 35 minutes.**

**What you practice here**

- Loading tokenizers from the Hub with `AutoTokenizer`.
- Turning text into token ids with the tokenizer's `encode`, and back into text with `decode` and `batch_decode`, and counting the characters, words and tokens of a text.
- Turning a messages list into the prompt a model really reads, with the tokenizer's chat template.
- Seeing that different models cut the same text into different tokens.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week3/day3", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson opens up tokenizers. Text becomes token ids and back, and an Instruct model's chat template turns the
messages list you have sent since Week 1 into one string, with special tokens around each message: that string, as
token ids, is what the model really reads. Look at a WrestleMania question the way three models see it.

transformers is not in the checks' Python, so `load_tokenizers` takes `AutoTokenizer` as its first parameter, and the
other functions take tokenizers it loaded: one tokenizer, or for `compare_tokenizers` a dict of them. They call these
as the lesson does. The checks hand them stand-in tokenizers with Llama 3's chat template, which cut text into words,
digits and marks.

## What each call should give

With the checks' stand-in tokenizer (each real tokenizer cuts the text its own way, and so counts differently):

```python
tokenizers = load_tokenizers(AutoTokenizer, ["meta-llama/Meta-Llama-3.1-8B-Instruct", "microsoft/Phi-4-mini-instruct"])
# {'meta-llama/Meta-Llama-3.1-8B-Instruct': <its tokenizer>, 'microsoft/Phi-4-mini-instruct': <its tokenizer>}

llama = tokenizers["meta-llama/Meta-Llama-3.1-8B-Instruct"]
token_counts(llama, "Hulk Hogan body-slammed André the Giant at WrestleMania III")
# {'characters': 59, 'words': 9, 'tokens': 13}

token_pieces(llama, "Hulk Hogan body-slammed André")
# ['<|begin_of_text|>', 'Hulk', ' Hogan', ' body', '-', 'slammed', ' Andr', 'é']

chat_prompt(llama, "You answer questions about WWE history.", "Who won the main event of WrestleMania III?")
# '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou answer questions about WWE history.<|eot_id|>'
# '<|start_header_id|>user<|end_header_id|>\n\nWho won the main event of WrestleMania III?<|eot_id|>'
# '<|start_header_id|>assistant<|end_header_id|>\n\n'

compare_tokenizers(tokenizers, "Hulk Hogan body-slammed André the Giant at WrestleMania III")
# {'meta-llama/Meta-Llama-3.1-8B-Instruct': 13, 'microsoft/Phi-4-mini-instruct': ...}
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `load_tokenizers(auto_tokenizer, model_names)`

Loads the tokenizer of each model you name. `auto_tokenizer` is transformers' `AutoTokenizer`. Load each name's
tokenizer once, in order, with its loading method, and return a dict from each name to its tokenizer, in the same
order.

<details><summary>Hint</summary>

```python
{name: auto_tokenizer.from_pretrained(name) for name in model_names}
```

</details>

In [ ]:
def load_tokenizers(auto_tokenizer, model_names):
    # Write load_tokenizers here, as described above.
    raise NotImplementedError("load_tokenizers is not written yet")

In [ ]:
check("load_tokenizers")

### 2 · `token_counts(tokenizer, text)`

Counts the characters, words and tokens of a text, as the lesson does. Encode the text into token ids with the
tokenizer, keeping its defaults, so that the token that marks the start of a text is counted, as in the lesson. Return
a dict with the keys `"characters"`, `"words"` and `"tokens"`: the length of the text, the number of words you get by
splitting it at spaces, and the number of token ids.

<details><summary>Hint</summary>

```python
tokens = tokenizer.encode(text)
counts = {"characters": len(text), "words": len(text.split(" ")), "tokens": len(tokens)}
```

</details>

In [ ]:
def token_counts(tokenizer, text):
    # Write token_counts here, as described above.
    raise NotImplementedError("token_counts is not written yet")

In [ ]:
check("token_counts")

### 3 · `token_pieces(tokenizer, text)`

Shows how a tokenizer cuts a text: the text of each of its tokens, in order. Encode the text into token ids, then
decode the ids back into text one by one, keeping the special tokens: a Llama tokenizer's list starts with
`"<|begin_of_text|>"`, and the pieces after it, joined, give the text back. Return the list of strings. Decode rather
than ask for the tokens themselves: a real tokenizer's tokens are byte-level pieces such as `"ĠHogan"`, not text.

<details><summary>Hint</summary>

```python
tokenizer.batch_decode(tokenizer.encode(text))
```

</details>

In [ ]:
def token_pieces(tokenizer, text):
    # Write token_pieces here, as described above.
    raise NotImplementedError("token_pieces is not written yet")

In [ ]:
check("token_pieces")

### 4 · `chat_prompt(tokenizer, system, question)`

Builds the prompt string a chat model really reads, from a system message and a question. Make the messages list, a
system message holding `system` and then a user message holding `question`, and apply the tokenizer's chat template to
it. Ask for text rather than token ids, and for the generation prompt, which ends the string with the cue for the
assistant's turn. Return the string.

<details><summary>Hint</summary>

```python
messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

</details>

In [ ]:
def chat_prompt(tokenizer, system, question):
    # Write chat_prompt here, as described above.
    raise NotImplementedError("chat_prompt is not written yet")

In [ ]:
check("chat_prompt")

### 5 · `compare_tokenizers(tokenizers, text)`

Counts how many tokens each tokenizer makes of the same text. `tokenizers` is a dict from a name to a tokenizer, as
`load_tokenizers` returns. Return a dict from each name to the number of ids that tokenizer gives when it encodes
`text`, in the same order.

<details><summary>Hint</summary>

```python
{name: len(tokenizer.encode(text)) for name, tokenizer in tokenizers.items()}
```

</details>

In [ ]:
def compare_tokenizers(tokenizers, text):
    # Write compare_tokenizers here, as described above.
    raise NotImplementedError("compare_tokenizers is not written yet")

In [ ]:
check("compare_tokenizers")

## Check everything

The cell below runs all 6 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 3, Day 3 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

This runs in your codespace: the lesson says a CPU is enough, and only the tokenizers download, a few MB each. Llama
3.1 is gated, so accept Meta's terms on its Hugging Face page first, with the account your token belongs to.

1. Add your Hugging Face token to `.env` as `HF_TOKEN=hf_...`, load it with `load_dotenv(override=True)`, and log in
   with `login(os.getenv("HF_TOKEN"))` from `huggingface_hub`.
2. Load three of the lesson's tokenizers with `load_tokenizers(AutoTokenizer, ...)`:
   `meta-llama/Meta-Llama-3.1-8B-Instruct`, `microsoft/Phi-4-mini-instruct` and `Qwen/Qwen2.5-Coder-7B-Instruct`.
3. For a WrestleMania sentence of your own, print `token_counts` and `token_pieces` with each tokenizer, then
   `compare_tokenizers`. Which model uses the fewest tokens?
4. Print `chat_prompt` with each tokenizer for a WWE history question, and spot what differs: Llama 3.1 adds two date
   lines to its system block, and Phi-4 and Qwen mark the messages with other special tokens.

<details><summary>Stuck? One way to write it</summary>

```python
import os

from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer

load_dotenv(override=True)
login(os.getenv("HF_TOKEN"))

NAMES = ["meta-llama/Meta-Llama-3.1-8B-Instruct", "microsoft/Phi-4-mini-instruct", "Qwen/Qwen2.5-Coder-7B-Instruct"]
SENTENCE = "Hulk Hogan body-slammed André the Giant in front of 93,173 fans at WrestleMania III"
tokenizers = load_tokenizers(AutoTokenizer, NAMES)
for name, tokenizer in tokenizers.items():
    print(name, token_counts(tokenizer, SENTENCE))
    print(token_pieces(tokenizer, SENTENCE))
print(compare_tokenizers(tokenizers, SENTENCE))

for name, tokenizer in tokenizers.items():
    print(f"--- {name}")
    print(chat_prompt(tokenizer, "You answer questions about WWE history.",
                      "Who won the main event of WrestleMania III?"))
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.